# Diachrone Frequenzdiagramme der Spanischen Grippe

Teil der Fallstudie „Quantitative Analyse der Medienwellen der Spanischen Grippe (1918/19)“

<img src="https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/bibliocon/assets/images/two_lines_combined.png">

## „In früheren Folgen“ (etwas Kontext)

Dieses Notebook ist nur ein Teil einer größeren didaktischen Fallstudie. Die konkrete Fallstudie widmet sich der folgenden Forschungsfrage:

> Lassen sich für die Spanische Grippe 1918/1919 mit Fokus auf den Berliner Raum Muster in der öffentlichen Aufmerksamkeit ausmachen, die eine wellenartige Verlaufsform aufweisen?

Weitere Informationen dazu finden sich im <a href="https://quadriga-dk.github.io/Text-Fallstudie-1/research_question/research-question_research-question.html" target="_blank"> Unterkapitel „Fragestellung“ </a>.

In den vorherigen Teilen dieser Fallstudie haben wir ein Korpus aus zwei Berliner Zeitungen der Jahre 1918 und 1919 zusammengestellt (siehe <a href="https://quadriga-dk.github.io/Text-Fallstudie-1/corpus_collection/corpus-collection_building-our-corpus.html"  target="_blank">„Aufbau des Forschungskorpus“</a>). Anschließend haben wir digitale Bilder mithilfe von OCR-Verfahren in Text umgewandelt (siehe <a href="https://quadriga-dk.github.io/Text-Fallstudie-1/ocr/ocr_intro.html" target="_blank">„OCR. Von Bild zu Text“</a>). Danach wurden die digitalisierten Texte mit dem NLP-Tool spaCy verarbeitet, um tokenisierte und lemmatisierte Texte zu erhalten (siehe <a href="https://quadriga-dk.github.io/Text-Fallstudie-1/corpus_processing/corpus-processing_intro.html" target="_blank">„Korpusverarbeitung. Von Strings zu Token“</a>.

In diesem Notebook setzen wir die Fallstudie mit einer quantitativen Exploration dieser Texte fort. Dabei versuchen wir, die „Medienwellen“ der Spanischen Grippe anhand kombinierter Worthäufigkeiten in verschiedenen Zeitabschnitten sichtbar zu machen.



## Übersicht über dieses Notebook 
Im Folgenden werden die von SpaCy annotierten Dateien (CSV-Format) analysiert. Unser Ziel ist es, die Wort-/Lemma-Häufigkeiten einer vordefinierten Wortgruppe für die Monate der Jahre 1918 und 1919 zu plotten und zu sehen, ob sie mit den Wellen der Grippepandemie korrelieren.
Dafür werden folgende Schritte durchgeführt:
1. Einlesen des Korpus, der Metadaten und der Grippe-Wortliste
2. Extraktion der Worthäufigkeiten und Plotten der Worthäufigkeiten
3. Diskussion der Ergebnisse

## Bevor die Action beginnt: kurze Einführung in Jupyter-Notebooks

Was Sie im Moment sehen, ist ein **Jupyter-Notebook**. Jupyter ermöglicht es Ihnen, Python-Code in Ihrem Browser zu schreiben und auszuführen. Es gibt Jupyter-Zellen mit Texten (wie diese hier) und Zellen mit Code. Notebook-Zellen mit Code sehen so aus:

In [ ]:
print('Hallo Welt!')

Um **Code in der Zelle auszuführen**, treten Sie darauf und drücken Sie **Cmd/Strg + Enter**. Eine weitere nützliche Tastenkombination ist **Shift+Enter**. Sie führt die Shell aus und geht zur nächsten über. Nützlich, wenn Sie durch viele Zellen klicken müssen. 

## Installation und Import der erforderlichen Python-Bibliotheken 
(ein technischer Schritt)

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
<b>Voraussetzungen zur Ausführung des Jupyter Notebooks</b>
<ol>
<li> Installieren der Bibliotheken </li>
<li> Pfad zu den Daten setzen</li>
<li> Laden der Daten (z.B. über den Command `wget` (s.u.))</li>
</ol>
Zum Testen: Ausführen der Zelle „load libraries“ und der Sektion „Einlesen der Daten“. </br>
Alle Zellen, die mit 🚀 gekennzeichnet sind, werden nur bei der Ausführung des Notebooks in Colab / JupyterHub bzw. lokal ausgeführt. 
</details>

In [ ]:
#  🚀 Install libraries 
! pip3 install pandas altair itables tqdm

In [ ]:
import re
import requests
from pathlib import Path
import pandas as pd
from itables import show
from tqdm.auto import tqdm

import altair as alt
alt.data_transformers.disable_max_rows()

## Einlesen der Daten, Metadaten und der Grippe-Wortliste

### Einlesen des Korpus (CSV-Dateien)

Dies ist der erste tatsächlich „inhaltliche“ Schritt dieses Teils der Forschungspipeline. Wir lesen die von spaCy verarbeiteten Texte in den aktuellen Arbeitsspeicher ein (in das Dictionary `corpus_annotations`).

Um eine/mehrere Dateien mit Python bearbeiten zu können, müssen die Dateien zuerst ausgewählt werden, d. h., der [Pfad](https://en.wikipedia.org/wiki/Path_(computing)) zu den Dateien wird gesetzt und anschließend werden die Dateien eingelesen. 


<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Zuerst wird der Ordner angelegt, in dem die CSV-Dateien gespeichert werden. Der Einfachheit halber wird die gleiche Datenablagestruktur wie in dem <a href="https://github.com/quadriga-dk/Text-Fallstudie-1/tree/main">GitHub Repository</a>, in dem die Daten gespeichert sind, vorausgesetzt. </br>
Danach werden alle CSV-Dateien im Korpus heruntergeladen und gespeichert. Dafür sind folgende Schritte nötig:
<ol>
    <li>Es wird eine Liste erstellt, die die URLs zu den einzelnen CSV-Dateien beinhaltet.</li>
    <li>Die Liste wird als txt-Datei gespeichert.</li>
    <li>Alle Dateien aus der Liste werden heruntergeladen und in dem Ordner <i>../data/csv</i> gespeichert.</li>
</ol>
Sollten die Dateien schon an einem anderen Ort vorhanden sein, können die Dateipfade zu den Ordnern angepasst werden. </br>
</details>

In [ ]:
# 🚀 Create data directory path
corpus_dir = Path("../data/csv")
if not corpus_dir.exists():
    corpus_dir.mkdir(parents = True, exist_ok = True)

In [ ]:
# 🚀 Create download list 
github_api_txt_dir_path = "https://api.github.com/repos/quadriga-dk/Text-Fallstudie-1/contents/data/csv"
txt_dir_info = requests.get(github_api_txt_dir_path).json()
url_list = [entry["download_url"] for entry in txt_dir_info]

# 🚀 Write download list as txt file
url_list_path = Path("github_csv_file_urls.txt")
with url_list_path.open('w') as output_txt:
    output_txt.write("\n".join(url_list))

In [ ]:
# ⚠️ Only execute, if you haven't downloaded the files yet!
# 🚀 Download all csv files – this step will take a while (ca. 7 minutes)
! wget -i github_csv_file_urls.txt -P ../data/csv

Setzen des Pfads:

In [ ]:
# set the path to csv files to be processed
csv_dir = Path(r"../data/csv")

Einlesen der CSV-Dateien in das Dictionary `corpus_annotations`

In [ ]:
# Create dictionary to save the corpus data (filenames and tables)
corpus_annotations = {}

# Iterate over csv files 
for file in csv_dir.iterdir():
    # check if the entry is a file, not a directory
    if file.is_file():
        # check if the file has the correct suffix csv
        if file.suffix == '.csv':
            # read the csv table to a data frame
            data = pd.read_csv(file) 
            # save the data frame to the dictionary, key=filename (without suffix), value=dataframe
            corpus_annotations[file.with_suffix("").name] = data

Wie viele Dateien wurden eingelesen?

In [ ]:
len(corpus_annotations)

Wie sieht der Anfang der ersten Datei aus?

In [ ]:
corpus_annotations[list(corpus_annotations.keys())[5]].head()

### Einlesen der Metadaten

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Der Pfad kann in der Variable <i>metadata_path</i> angepasst werden. Die einzulesende Datei muss die Endung `.csv` haben. </br>
</details>

In [ ]:
# 🚀 Create metadata directory path
metadata_dir = Path("../data/metadata")
if not metadata_dir.exists():
    metadata_dir.mkdir()

In [ ]:
# 🚀 Load the metadata file from GitHub 
! wget https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/data/metadata/QUADRIGA_FS-Text-01_Data01_Corpus-Table.csv -P ../data/metadata

In [ ]:
# set path to metadata file
metadata_path = '../data/metadata/QUADRIGA_FS-Text-01_Data01_Corpus-Table.csv'

# read metadata file to pandas dataframe
corpus_metadata = pd.read_csv(metadata_path, sep=';')
corpus_metadata['DC.date'] = pd.to_datetime(corpus_metadata['DC.date'])
#corpus_metadata = corpus_metadata.set_index('DC.identifier')

Wie sieht die Metadaten-Datei aus?

In [ ]:
show(corpus_metadata)

Die Zeilen in der obigen Tabelle sind Zeitungsausgaben. Für jede Ausgabe sehen wir ihre Annotationsdatei – sie kann über `DC.identifier` aus dem Dictionary `corpus_annotations` abgerufen werden. Schauen wir uns zum Beispiel den Anfang der Ausgabe der Berliner Morgenpost vom 3. Januar 1918 an:


In [ ]:
corpus_annotations['SNP2719372X-19180103-0-0-0-0'].head(10)

### Das semantische Feld der Spanischen Grippe

#### Erläuterung: Semantisches Feld
Das Ziel der Analyse ist es, zu quantifizieren, wie viel über die Spanische Grippe berichtet wird. Dafür sollen möglichst alle und nur die Textstellen erfasst werden, in denen die Spanische Grippe erwähnt wird. Eine Erwähnung liegt dann vor, wenn ein Wort vorkommt, das mit der Spanischen Grippe im Zusammenhang steht. Die Sammlung dieser Wörter nennen wir Semantisches Feld. Da die Wörter losgelöst von ihrem Kontext analysiert werden, sollten sie so gewählt sein, dass sie sich auf die Spanische Grippe und nur auf diese beziehen.

#### Erstellung des semantischen Felds
Da <a href="https://en.wikipedia.org/wiki/Large_language_model" class="external-link" target="_blank">Large Language Models</a> sehr gut dazu in der Lage sind, semantisch ähnliche Wörter zu erzeugen, haben wir das semantische Feld mit Hilfe des Chatbots <a href="https://openai.com/index/chatgpt/" class="external-link" target="_blank">ChatGPT</a> erstellt.

```{admonition} Spezifikation zur ChatGPT-Nutzung
:class: hinweis
**Verwendete ChatGPT-Version**: 1.2024.157 (1718053765), Modell: ChatGPT 4o

**Prompt**:

Du bist eine Digital Humanities-Forscherin mit Expertise im Bereich der Computerlinguistik. Du verfügst über umfassende Kenntnisse im Bereich der Semantik und des Text und Data Mining.

Bitte erstelle ein semantisches Feld zum Thema „Grippe“. Die Sprache ist deutsch. Bedingungen für die Wörter des semantischen Feldes sind:
* die Wörter sollen Substantive sein;
* Komposita sind erlaubt;:
* die Wörter sollen sich am historischen Sprachgebrauch der Jahre 1918 und 1919 orientieren;
* die Wörter sollen spezifisch für den Kontext „Grippe“ sein;
* die Wörter sollen nicht mehrdeutig sein, also nach Möglichkeit nicht in anderen semantischen Kontext vorkommen;
```
Als Resultat haben wir eine Liste mit 25 Nomen erhalten:
Influenza, Grippe, Grippeepidemie, Grippewelle, Grippekranke, Grippepandemie, Lungenentzündung, Krankheitswelle, Seuchenzug, Krankheitsausbruch, Fieberanfall, Schüttelfrost, Atemnot, Körpererschöpfung, Genesungszeit, Ansteckungsgefahr, Seuchenschutz, Desinfektionsmittel, Schutzmaske, Krankenstation, Isolationsstation, Sanitätsdienst, Krankheitsverlauf, Todesopfer, Krankheitssymptom, Erkrankungsfall, Lungeninfektion


#### Einlesen der Wortliste (Semantisches Feld „Grippe“)

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
In der folgenden Codezelle legen wir den Pfad zur Textdatei fest, in der die Liste gespeichert ist. Anschließend lesen wir den Text aus der Datei und teilen ihn in Wörter auf (anhand von Zeilenumbrüchen).
</details>

In [3]:
# 🚀 Create word list directory path
wordlist_dir = Path("../data/wordlist")
if not wordlist_dir.exists():
    wordlist_dir.mkdir()

In [ ]:
# 🚀 Load the wordlist file from GitHub 
! wget https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/data/wordlist/grippe.txt -P ../data/wordlist

In [ ]:
path_to_wordlist = Path("../data/wordlist/grippe.txt")
word_list = path_to_wordlist.read_text().split("\n")

Wie sieht die Wortliste aus?

In [ ]:
word_list

## Suche nach einem Lemma und plotte die Häufigkeit (pro Tag, Woche und Monat)

### Warum die Häufigkeit analysieren?
Die Analyse von Worthäufigkeiten ist sowohl in der Korpuslinguistik als auch in den Digital Humanities weit verbreitet. Für die Analyse von Inhaltswörtern (Nomen, Verben, Adjektive, Adverben) wird angenommen, dass ein hohes Vorkommen mit der Wichtigkeit der Wörter im Text korreliert. Besonders bei einer vergleichenden Analyse (etwa von zwei Zeitungen oder einem Thema über Zeit) ist die Häufigkeitsanalyse sinnvoll, da der Vergleich so quantisierbar wird. 

### Häufigkeit von Grippe
Um die Wichtigkeit eines Themenfelds wie der Spanischen Grippe zu untersuchen, bietet es sich an, nicht nur das Vorkommen eines einzelnen Wortes wie „Grippe“ zu untersuchen, sondern andere, mit Grippe im Zusammenhang stehende Wörter zu sammeln. Die Wörter werden in der Grundform angegeben, sodass sie mit den Lemmata im Text verglichen werden können.
Für jedes Wort wird dann die Häufigkeit errechnet, diese nennt sich **absolute Häufigkeit**. Die absoluten Häufigkeiten werden addiert, sodass sich pro Text eine Zahl ergibt, die die Summe aller Häufigkeiten der Grippenbezogenen Wörter angibt.

`````{admonition} Beispiel
:class: hinweis
1. **Text**: Die Grippe wütet weiter. Zunahme der schweren Fälle in Berlin. Die Zahl der Grippefälle ist in den letzten Tagen auch in Groß-Berlin noch erheblich gestiegen. Die Warenhäuser und sonstigen großen Geschäfte, die Kriegs- und die privaten Betriebe klagen, daß übermäßig viele Angestellte sich haben krank melden müssen und auch bei der Post und bei der Straßenbahn ist der Prozentsatz der Grippekranken deutlich gestiegen. 
2. **Lemmatisierter Text**: der Grippe wüten weiter -- Zunahme der schwer Fall in Berlin -- der Zahl der Grippefall sein in der letzter Tag auch in Groß-Berlin noch erheblich steigen -- der Warenhaus und sonstig groß Geschäft -- der Krieg und der privat Betrieb klagen -- daß übermäßig vieler angestellter sich haben krank melden müssen und auch bei der Post und bei der Straßenbahn sein der Prozentsatz der Grippekranke deutlich steigen --
3. **Semantisches Feld** „Grippe“: Grippe, Grippefall, Grippekranke
4. **Häufigkeitsanalyse**: 

```{table}
:name: Häufigkeitsanalyse
| Wort  | Häufigkeit| 
|--------|-------|
| Grippe    | 1   |
| Grippefall | 1 |
| Grippekranke  | 1 |
```
5. **Summierte Häufigkeit**: 3

`````

### Vergleichbarkeit von Häufigkeiten
Für die Vergleichbarkeit von Worthäufigkeiten in Texten ist wichtig, dass die Texte auch ansonsten vergleichbar sind. Stammen die Texte z. B. aus unterschiedlichen Zeiträumen müssten ggf. zeitspezifische semantische Felder erstellt werden, um für den Sprachwandel Rechnung zu tragen. Auch sollten die Texte eine ähnliche Länge haben, sodass eine erhöhte Häufigkeit tatsächlich auf eine erhöhte Wichtigkeit zurückgeführt werden kann.
Wenn Texte verschieden lang sind, sollten die Häufigkeiten **normalisiert** werden, das heißt sie werden in Bezug zur Textlänge gesetzt. Dafür wird die absolute Häufigkeit durch die Textlänge dividiert, daraus ergibt sich die **relative Frequenz**. Die relative Frequenz des semantischen Felds „Grippe“ kann als Anteil der Grippewörter am Gesamttext gesehen werden. 

`````{admonition} Beispiel
:class: hinweis
Der Beispieltext besteht aus insgesamt 69 Wörter, davon sind 4 Wörter in dem semantischen Feld „Grippe“ vorhanden. Daraus ergibt sich folgende Rechnung:

$ f = {4 \over 69} = {0.05797101449} $.

Das heißt: Jedes zwanzigste Wort im Text steht im Zusammenhang mit der Spanischen Grippe. 
`````

### Analyse des gesamten Korpus 
Um den Verlauf der Aufmerksamkeit nachzuvollziehen, wird für jeden Text im Korpus die relative Frequenz des semantischen Felds „Grippe“ berechnet und in einer Tabelle gespeichert. Die Frequenzen werden dann über die Zeit verglichen. 

### Back to Action. Was jetzt passiert:

1. Datum zu den Annotationen hinzufügen
2. Annotationen in einer Datenstruktur (einem DataFrame) speichern
3. Lemmata suchen und nach Zeitabschnitt gruppieren
4. Häufigkeiten plotten

### Datum zu den Annotationen hinzufügen

Daten wieder mit Metadaten verknüpfen

In [ ]:
def add_date_to_corpus_annotations(corpus_metadata: pd.DataFrame, corpus_annotated: dict[str, pd.DataFrame]) -> None:
    """Add date colum from corpus_metadata to corpus annotated. Map by DC.identifier / filename."""
    for identifier, df in corpus_annotated.items():
        if identifier in corpus_metadata["DC.identifier"].values:
            df["date"] = corpus_metadata[corpus_metadata["DC.identifier"] == identifier]["DC.date"].item()

In [ ]:
add_date_to_corpus_annotations(corpus_metadata, corpus_annotations)

Schauen wir uns an, wie die Annotation jetzt aussieht:

In [ ]:
corpus_annotations['SNP2719372X-19180103-0-0-0-0'].head(10)

Jedem Wort ein Datum hinzuzufügen mag redundant erscheinen, aber da wir als Nächstes für praktische Zwecke alle Annotationen zu einer riesigen Tabelle zusammenfügen werden, ist das tatsächlich ziemlich nützlich.

### Annotationen in einer Datenstruktur (einem DataFrame) speichern

In [ ]:
corpus_annotations_merged = pd.concat(corpus_annotations.values())

### Berechnen der absoluten und relativen Häufigkeiten zusammengefasst pro Tag, Woche und Monat

In [ ]:
def search_split_by_timeframe(merged_df: pd.DataFrame, search_terms=word_list) -> tuple[dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    """Get lemmata count of words in search_terms by month, week and days.
    :param pd.DataFrame merged_df: The merged dataframe of all annotations
    :param list search_terms: List of words to search in merged_df
    :return tuple: Two dictionaries with identical keys, saving the absolute and relative frequencies by three time frames respectively
    """
    # Filter dataframe by lemmata in word_list
    result = merged_df.query(f'Lemma.isin({search_terms})')

    # Collect lemmata count by time frames: month, week, day
    frequency_parameters = ["M", "W-MON", "D"]
    absolute_frequencies = {}
    relative_frequencies = {}
    for fp in tqdm(frequency_parameters, desc="Berechne Häufigkeiten", leave=False):
        # count absolute frequencies per each month/day/year
        absolute_frequencies[fp] = (
            result.groupby(pd.PeriodIndex(result['date'], freq=fp)).count().Lemma
        )
        # count relative frequencies per each month/day/year
        relative_frequencies[fp] = (
            absolute_frequencies[fp]
            / merged_df.groupby(pd.PeriodIndex(merged_df['date'], freq=fp)).count().Lemma.fillna(0)
        )

    return absolute_frequencies, relative_frequencies

Berechnen wir die Häufigkeiten einmal

In [ ]:
calculated_freqs = search_split_by_timeframe(corpus_annotations_merged)

Schauen wir uns an, wie die monatlichen Häufigkeiten aussehen:

In [ ]:
calculated_freqs[1]['M']

### Erstellen eines interaktiven Liniendiagramms

In diesem Schritt werden die extrahierten absoluten oder relativen Häufigkeiten mit einem Liniendiagramm visualisiert. Liniendiagramme eignen sich gut, um zeitliche Verläufe darzustellen, da lokale und globale Minima und Maxima leicht erkennbar sind und sie die Kontinuität der Daten unterstreichen. 

<img src="https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/assets/images/Drei-Wellen-1918-19-UK.png">

*Drei Wellen der Spanischen Grippe im Vereinigten Königreich. Quelle: Taubenberger, J. K., & Morens, D. M. (2006). 1918 Influenza: the Mother of All Pandemics. Emerging Infectious Diseases, 12(1), 15-22. https://doi.org/10.3201/eid1201.050979*

**Das Liniendiagramm** (siehe Abbildung oben), das wir als Orientierungsgröße für ein wellenartiges Muster nehmen, zeigt den zeitlichen Verlauf der durch die Spanische Grippe verursachten Todesfälle in Großbritannien. Die Ticks auf der x-Achse zeigen die Wochen und die Beschriftung die Monate an (in Form von Monat/Tag). In einer weiteren Beschriftung wird auf das Jahr verwiesen. Da die einzelnen Erhebungen (z. B. das erste lokale Minimum zwischen Juni und Juli 1918) granularer sind als die angegeben Monate, können wir annehmen, dass die Daten wochenweise zusammengefasst wurden. Die y-Achse gibt, wie die Beschriftung sagt, die Tode pro 1.000 Personen an.
Wir erstellen eine ähnliche Visualisierung mit dem Unterschied, dass die y-Achse die relative Worthäufigkeit angibt.

Die Häufigkeiten über Zeit ließen sich auch in einem Balkendiagramm darstellen. Diese sind nützlich, um Häufigkeiten in diskreten Zeitintervallen zu visualisieren, sie eignen sich aber weniger gut, um eine zeitliche Entwicklung zu zeigen. 


In [ ]:
import textwrap

def plot_frequencies(merged_df: pd.DataFrame, search_terms=word_list) -> alt.Chart:
    """
    Plot lemmata frequencies of words in search_terms over time, interactively.
    Two Altair dropdowns let the user switch between monthly/weekly/daily
    aggregation and absolute/relative counts.
    :param pd.DataFrame merged_df: The merged dataframe of all annotations
    :param list search_terms: List of words to search in merged_df
    """
    # Get the data -- here we actually count the frequencies
    absolute_frequencies, relative_frequencies = search_split_by_timeframe(
        merged_df, search_terms=search_terms
    )

    # Flatten the six (aggregation × type) series into one long-format DataFrame.
    # Altair expects tidy data; the two dropdowns will filter it down to one
    # (aggregation, type) combination at a time.
    agg_labels = {"M": "Monatlich", "W-MON": "Wöchentlich", "D": "Täglich"}
    rows = []
    for freq_code, agg_label in agg_labels.items():
        for typ_label, freq_dict in [
            ("Absolut", absolute_frequencies),
            ("Relativ", relative_frequencies),
        ]:
            series = freq_dict[freq_code]
            for period, value in series.items():
                rows.append({
                    "Datum": period.to_timestamp(),
                    "Wert": float(value) if pd.notna(value) else None,
                    "Aggregation": agg_label,
                    "Frequenztyp": typ_label,
                })
    df = pd.DataFrame(rows)

    # Two dropdowns, bound to selection_point params and used in transform_filter.
    agg_select = alt.selection_point(
        fields=["Aggregation"],
        bind=alt.binding_select(
            options=["Monatlich", "Wöchentlich", "Täglich"],
            name="Zeitraum: ",
        ),
        value=[{"Aggregation": "Monatlich"}],
    )
    typ_select = alt.selection_point(
        fields=["Frequenztyp"],
        bind=alt.binding_select(
            options=["Absolut", "Relativ"],
            name="Häufigkeitstyp: ",
        ),
        value=[{"Frequenztyp": "Absolut"}],
    )

    # Wrap the search-term list into multiple subtitle lines so the title
    # doesn't force the chart wider than its 750 px content area.
    subtitle_lines = textwrap.wrap(", ".join(search_terms), width=90)

    print("Diagramm wird gerendert …")
    return (
        alt.Chart(df)
        .mark_line(strokeWidth=2)
        .encode(
            x=alt.X("Datum:T", title="Zeit"),
            y=alt.Y("Wert:Q", title="Häufigkeit"),
            tooltip=[
                alt.Tooltip("Datum:T", title="Datum"),
                alt.Tooltip("Wert:Q", title="Häufigkeit", format=".4f"),
                alt.Tooltip("Aggregation:N"),
                alt.Tooltip("Frequenztyp:N"),
            ],
        )
        .add_params(agg_select, typ_select)
        .transform_filter(agg_select)
        .transform_filter(typ_select)
        .properties(
            width=750,
            height=400,
            title=alt.TitleParams(
                text="Häufigkeit folgender Wörter:",
                subtitle=subtitle_lines,
                subtitleFontSize=11,
                anchor="start",
            ),
        )
    )

In [ ]:
# Call the function to plot the frequencies 
plot_frequencies(corpus_annotations_merged, search_terms=word_list)

### Worteingabe für die Suche

Hier können Sie versuchen, Ihre eigenen Wortlisten einzufügen (z. B. geben Sie einmal „Krieg, Tod“ ein und schauen Sie, wie die Plotlinie jetzt aussieht). Drücken Sie dann die **Eingabetaste**

In [ ]:
text_input = input("Geben Sie die zu suchenden Wörter ein und trennen Sie sie durch Kommas, wenn es mehrere sind, und Drücken Sie die Eingabetaste:")
# Convert the input to a list by splitting the input by comma
text_input = [x.strip() for x in text_input.split(',')]
plot_frequencies(corpus_annotations_merged, search_terms=text_input)

## Diskussion des Zwischenergebnisses

In Taubenberger & Morens (2006) wird festgestellt, dass 'The first pandemic influenza wave appeared in the spring of 1918, followed in rapid succession by much more fatal second and third waves in the fall and winter of 1918–1919, respectively' ('Die erste pandemische Influenza-Welle im Frühjahr 1918 auftrat, gefolgt von weitaus tödlicheren zweiten und dritten Wellen im Herbst und Winter 1918–1919'). Sie ergänzen diese Aussage auch mit einem Diagramm aus einem früheren Papier (Jordan 1927):

<img src="https://wwwnc.cdc.gov/eid/images/05-0979-F1.gif">

Unsere zwei Wellen der Erwähnungen des Wortes 'Grippe' scheinen den Sterblichkeitszahlen zu entsprechen, was darauf hindeuten könnte, dass die Methode, obwohl sehr einfach, funktioniert und dass historische Ereignisse manchmal in Wortfrequenzzählungen reflektiert werden können... Die dritte Welle scheint nicht reproduziert zu werden, was eine weitere Untersuchung erfordert. Eine Hypothese könnte sein, dass, ähnlich wie bei der COVID-Pandemie, neue Krankheitswellen irgendwann aufhören, die Aufmerksamkeit der Öffentlichkeit zu erregen. Beispielsweise waren die COVID-Wellen im Jahr 2021 stärker als die im Jahr 2020, aber die Berichterstattung in den Nachrichten nahm bereits ab. Dies könnte besonders für Anfang 1919 zutreffen, als nach dem Verlust des Krieges und der Revolution von 1918 Grippetodesfälle kein Nachrichtenthema mehr waren.

### Bibliographie
* Jordan E. (1927). Epidemic influenza: a survey. Chicago: American Medical Association.
* Taubenberger, J. K., & Morens, D. M. (2006). 1918 Influenza: the Mother of All Pandemics. Emerging Infectious Diseases, 12(1), 15-22. https://doi.org/10.3201/eid1201.050979